# Partial Fractions over $\mathbb{C}$

In calculus, partial fractions over $\mathbb{R}$ come in two flavours: terms $\frac{A}{x - a}$ for real roots of the denominator and awkward terms $\frac{Bx + C}{x^2 + px + q}$ for irreducible quadratics. Over $\mathbb{C}$ the second kind disappears. By the fundamental theorem of algebra every denominator splits into **linear** factors, so every proper rational function is a sum of terms $\frac{A}{(z - z_k)^j}$. The coefficients of the simple terms come from a one-line **cover-up formula**, $A_k = P(z_k)/Q'(z_k)$, which is the first appearance of what module 06 calls a **residue**. This notebook develops the complex decomposition, shows how conjugate pairs recombine into the familiar real quadratic terms, uses sympy's `apart`, and applies the method to an integral.

This is notebook 04 of module 02 (**Polynomials and roots**) in the **Learning Complex Numbers** series.

## 1. Introduction

Partial fractions turn a complicated rational function into a sum of the simplest possible pieces, each with a single **pole** (a point where the function blows up). That decomposition is the workhorse behind:

- integrating rational functions (each piece integrates to a logarithm or a power);
- inverse Laplace and $z$-transforms in control and signal processing (each pole $p$ becomes a mode $e^{pt}$ or $p^n$);
- the residue theorem (module 06), where the coefficient $A_k$ of $\frac{1}{z - z_k}$ is all that survives a contour integral.

Over $\mathbb{R}$ you must handle linear and quadratic factors separately. Over $\mathbb{C}$ there is one uniform rule, and the real version can always be recovered by pairing conjugate terms.

**Prerequisites**

- Partial fraction expansion over $\mathbb{R}$, and integration of the pieces: [Partial fraction expansion](../../../LearningCalculus/hypersklill/04_integrals/06_partial_fraction_expansion.ipynb) (LearningCalculus).
- Complete factorization of polynomials over $\mathbb{C}$ and over $\mathbb{R}$: [The fundamental theorem of algebra](02_fundamental_theorem_of_algebra.ipynb).
- Conjugate-root theorem and real quadratic factors $z^2 - 2\operatorname{Re}(w)z + |w|^2$: [Quadratics and conjugate roots](01_quadratics_and_conjugate_roots.ipynb).
- Complex division: [Conjugate, modulus and division](../00_foundations/03_conjugate_modulus_and_division.ipynb).

After completing this notebook you will be able to:

- Decompose any rational function over $\mathbb{C}$, including repeated poles and improper fractions.
- Compute coefficients of simple poles with the cover-up formula $A_k = P(z_k)/Q'(z_k)$.
- Recombine conjugate pairs into real terms $\frac{Bx + C}{x^2 + px + q}$.
- Use `sympy.apart` with `extension=` and `full=True`, and use complex partial fractions to evaluate a real integral.

## 2. Theory

### 2.1 Setting

A **rational function** is $R(z) = \frac{P(z)}{Q(z)}$ with polynomials $P, Q$, $Q \ne 0$, and (after cancelling) no common roots. It is **proper** if $\deg P < \deg Q$. If not, polynomial division gives $P = SQ + T$ with $\deg T < \deg Q$, so

$$
\frac{P}{Q} = S + \frac{T}{Q},
$$

a polynomial plus a proper rational function. From now on, $R$ is proper.

By the FTA, $Q(z) = c\,(z - z_1)^{m_1}\cdots(z - z_r)^{m_r}$ with distinct $z_k$. The $z_k$ are the **poles** of $R$; $m_k$ is the **order** of the pole.

### 2.2 The decomposition theorem

> **Theorem.** A proper rational function $R = P/Q$ with $Q$ factored as above can be written uniquely as
> $$ R(z) = \sum_{k=1}^{r} \sum_{j=1}^{m_k} \frac{A_{k,j}}{(z - z_k)^j}, \qquad A_{k,j} \in \mathbb{C}. $$

*Why it works (sketch).* Count unknowns: there are $m_1 + \dots + m_r = \deg Q$ coefficients. Multiplying through by $Q$ turns the identity into an equation between polynomials of degree $< \deg Q$, which is a linear system of $\deg Q$ equations. One shows that the only decomposition of the zero function has all coefficients zero (look at the most singular term at each pole), so the square system is invertible.

The terms belonging to a single pole $z_k$,

$$
\frac{A_{k,m_k}}{(z - z_k)^{m_k}} + \dots + \frac{A_{k,1}}{z - z_k},
$$

form the **principal part** of $R$ at $z_k$: everything that blows up there. Subtracting it leaves a function that is finite at $z_k$.

### 2.3 Simple poles: the cover-up formula

If $z_k$ is a simple pole ($m_k = 1$), multiply $R$ by $(z - z_k)$ and let $z \to z_k$. Every other term is killed by the factor $(z - z_k)$, so

$$
A_k = \lim_{z \to z_k} (z - z_k) R(z) = \frac{P(z_k)}{\prod_{j \ne k}(z_k - z_j)}.
$$

That is Heaviside's **cover-up rule**: cover the factor $(z - z_k)$ in the denominator and evaluate the rest at $z_k$. An equivalent form that does not require factoring $Q$: since $Q(z) = (z - z_k)\,\tilde Q(z)$ gives $Q'(z_k) = \tilde Q(z_k)$,

$$
A_k = \frac{P(z_k)}{Q'(z_k)}.
$$

In module 06 this number is the **residue** of $R$ at $z_k$, written $\operatorname{Res}_{z = z_k} R$.

If all poles are simple:

$$
\frac{P(z)}{Q(z)} = \sum_{k=1}^{n} \frac{P(z_k)}{Q'(z_k)}\,\frac{1}{z - z_k}.
$$

### 2.4 Repeated poles

For a pole of order $m$, let $g(z) = (z - z_k)^m R(z)$, which is finite at $z_k$. Then $g(z) = A_{k,m} + A_{k,m-1}(z - z_k) + \dots$, a Taylor expansion, so

$$
A_{k,\,m-j} = \frac{1}{j!}\, g^{(j)}(z_k), \qquad j = 0, 1, \dots, m - 1.
$$

In particular the coefficient of the most singular term is $A_{k,m} = g(z_k)$ (cover-up again), and the coefficient of $\frac{1}{z - z_k}$, which will be the residue, is $\frac{1}{(m-1)!}g^{(m-1)}(z_k)$.

### 2.5 Real rational functions: conjugate coefficients and recombination

If $P$ and $Q$ have real coefficients, the poles come in conjugate pairs (notebook 01), and so do the coefficients:

$$
A_{\bar w} = \frac{P(\bar w)}{Q'(\bar w)} = \frac{\overline{P(w)}}{\overline{Q'(w)}} = \overline{A_w}.
$$

The pair of terms recombines to a real fraction. With $w = p + qi$ and $A = \alpha + \beta i$, put both over the common denominator $(x - w)(x - \bar w) = x^2 - 2px + p^2 + q^2$:

$$
\frac{A}{x - w} + \frac{\bar A}{x - \bar w}
= \frac{A(x - \bar w) + \bar A(x - w)}{(x - w)(x - \bar w)}
= \frac{2\operatorname{Re}(A)\, x - 2\operatorname{Re}(A\bar w)}{x^2 - 2\operatorname{Re}(w)\, x + |w|^2}.
$$

The numerator is real because $A\bar w + \bar A w = 2\operatorname{Re}(A\bar w)$. This is exactly the $\frac{Bx + C}{x^2 + px + q}$ term from real partial fractions, with

$$
B = 2\operatorname{Re}(A), \qquad C = -2\operatorname{Re}(A\bar w).
$$

### 2.6 Application: integrating rational functions

For real $x$ and a non-real pole $w$, the complex number $x - w$ has constant imaginary part $-\operatorname{Im} w \ne 0$, so as $x$ moves along the real line it never crosses the negative real axis. On such a path the principal logarithm $\operatorname{Log}$ is a valid antiderivative:

$$
\frac{d}{dx}\operatorname{Log}(x - w) = \frac{1}{x - w}, \qquad\text{so}\qquad \int_a^b \frac{A\,dx}{x - w} = A\big[\operatorname{Log}(b - w) - \operatorname{Log}(a - w)\big].
$$

(For a real pole outside $[a, b]$, $x - w$ keeps a constant sign and the formula reduces to the familiar $\ln|x - w|$.) The complex logarithm and its branch cut are treated properly in [module 03](../03_complex_functions/03_the_complex_logarithm_and_branch_cuts.ipynb); here we only need that $\operatorname{Log} z = \ln|z| + i\operatorname{Arg} z$ is differentiable away from the negative real axis. So **every** rational integral reduces to evaluating logarithms at complex points. For example,

$$
\frac{1}{x^2 + 1} = \frac{1}{2i}\left(\frac{1}{x - i} - \frac{1}{x + i}\right)
\quad\Longrightarrow\quad
\int \frac{dx}{x^2 + 1} = \frac{1}{2i}\Big(\operatorname{Log}(x - i) - \operatorname{Log}(x + i)\Big) + C.
$$

Since $|x - i| = |x + i|$, the real parts cancel; with $\operatorname{Arg}(x + i) = \frac{\pi}{2} - \arctan x$ and $\operatorname{Arg}(x - i) = -\operatorname{Arg}(x + i)$, the bracket is $-2i\left(\frac{\pi}{2} - \arctan x\right)$, so the antiderivative is $\arctan x - \frac{\pi}{2}$. That is $\arctan x$ plus a constant: the arctangent is a logarithm in disguise.

## 3. Worked Examples

### Example 1: $\dfrac{1}{z^2 + 1}$

Poles $\pm i$, both simple. With $P = 1$, $Q' = 2z$:

$$
A_i = \frac{1}{2i} = -\frac{i}{2}, \qquad A_{-i} = \frac{1}{-2i} = \frac{i}{2},
\qquad\text{so}\qquad
\frac{1}{z^2 + 1} = \frac{-i/2}{z - i} + \frac{i/2}{z + i}.
$$

The coefficients are conjugate, as 2.5 predicts. Check: $\frac{-i/2}{z - i} + \frac{i/2}{z + i} = \frac{-\frac{i}{2}(z + i) + \frac{i}{2}(z - i)}{z^2 + 1} = \frac{\frac12 + \frac12}{z^2 + 1}$ ✓.

### Example 2: $\dfrac{z + 3}{(z - 1)(z^2 + 4)}$ by cover-up, then recombined

Poles $1, 2i, -2i$, all simple.

- $A_1$: cover $(z - 1)$, evaluate at $1$: $\frac{1 + 3}{1 + 4} = \frac45$.
- $A_{2i}$: cover $(z - 2i)$ in $(z - 1)(z - 2i)(z + 2i)$, evaluate at $2i$:
  $$A_{2i} = \frac{3 + 2i}{(2i - 1)(4i)} = \frac{3 + 2i}{-8 - 4i} = -\frac{(3 + 2i)(8 - 4i)}{80} = -\frac{32 + 4i}{80} = -\frac25 - \frac{1}{20}i.$$
- $A_{-2i} = \overline{A_{2i}} = -\frac25 + \frac{1}{20}i$.

So over $\mathbb{C}$:
$$
R(z) = \frac{4/5}{z - 1} + \frac{-\frac25 - \frac{i}{20}}{z - 2i} + \frac{-\frac25 + \frac{i}{20}}{z + 2i}.
$$

Recombine the pair with $w = 2i$: $B = 2\operatorname{Re}(A) = -\frac45$; $A\bar w = \left(-\frac25 - \frac{i}{20}\right)(-2i) = \frac45 i + \frac{1}{10}i^2 = -\frac{1}{10} + \frac45 i$, so $C = -2\operatorname{Re}(A\bar w) = \frac15$. Over $\mathbb{R}$:
$$
R(x) = \frac{4/5}{x - 1} + \frac{-\frac45 x + \frac15}{x^2 + 4}.
$$

Check: $\frac45(x^2 + 4) + \left(-\frac45 x + \frac15\right)(x - 1) = \frac45x^2 + \frac{16}{5} - \frac45x^2 + \frac45x + \frac15x - \frac15 = x + 3$ ✓.

### Example 3: a repeated pole, $\dfrac{1}{(z^2 + 1)^2}$

Poles $\pm i$, each of order $2$. At $i$: $g(z) = (z - i)^2 R(z) = (z + i)^{-2}$.

- $A_{i,2} = g(i) = \frac{1}{(2i)^2} = -\frac14$.
- $A_{i,1} = g'(i) = -2(z + i)^{-3}\big|_{z = i} = \frac{-2}{(2i)^3} = \frac{-2}{-8i} = \frac{1}{4i} = -\frac{i}{4}$.

By conjugate symmetry $A_{-i,2} = -\frac14$ and $A_{-i,1} = \frac{i}{4}$:
$$
\frac{1}{(z^2 + 1)^2} = \frac{-i/4}{z - i} - \frac{1/4}{(z - i)^2} + \frac{i/4}{z + i} - \frac{1/4}{(z + i)^2}.
$$

### Example 4: an improper fraction, $\dfrac{z^3}{z^2 + 1}$

Divide first: $z^3 = z(z^2 + 1) - z$, so $\frac{z^3}{z^2 + 1} = z - \frac{z}{z^2 + 1}$. For the proper part, $A_{\pm i} = \frac{P(\pm i)}{Q'(\pm i)} = \frac{\pm i}{\pm 2i} = \frac12$:
$$
\frac{z^3}{z^2 + 1} = z - \frac{1/2}{z - i} - \frac{1/2}{z + i}.
$$

### Example 5: all $n$-th roots of unity at once

$Q(z) = z^n - 1$ has the simple poles $\omega^k$, $\omega = e^{2\pi i/n}$, and $Q'(z) = nz^{n-1}$. Since $(\omega^k)^n = 1$, $\frac{1}{n(\omega^k)^{n-1}} = \frac{\omega^k}{n}$, and

$$
\frac{1}{z^n - 1} = \frac1n\sum_{k=0}^{n-1} \frac{\omega^k}{z - \omega^k}.
$$

The cover-up formula in the form $P/Q'$ made this a one-line computation without ever writing out $\prod_{j \ne k}(\omega^k - \omega^j)$.

## 4. Python Examples

### 4.1 Cover-up coefficients numerically

For a denominator with simple roots, `np.roots` gives the poles, `np.polyder` gives $Q'$, and the coefficients are `polyval(P, z_k) / polyval(Q', z_k)`. We rebuild $R$ from the pieces and compare at random complex points.

In [ ]:
import numpy as np


def simple_partial_fractions(numerator, denominator):
    poles = np.roots(denominator)
    coefficients = np.polyval(numerator, poles) / np.polyval(np.polyder(denominator), poles)
    return poles, coefficients


numerator = [1, 3]                                 # z + 3
denominator = np.polymul([1, -1], [1, 0, 4])       # (z - 1)(z^2 + 4)
poles, coefficients = simple_partial_fractions(numerator, denominator)
for pole, coefficient in zip(poles, coefficients):
    print(f"pole {np.round(pole, 6)!s:>12}:  A = {np.round(coefficient, 6)}")

rng = np.random.default_rng(0)
test_points = rng.normal(size=5) + 1j * rng.normal(size=5)
direct = np.polyval(numerator, test_points) / np.polyval(denominator, test_points)
from_pieces = sum(A / (test_points - z_k) for z_k, A in zip(poles, coefficients))
print("\nmax |R - sum of pieces| at random points:", np.max(np.abs(direct - from_pieces)))

### 4.2 `sympy.apart`: over $\mathbb{Q}$, over $\mathbb{Q}(i)$, and fully

By default `apart` keeps coefficients rational, so irreducible quadratics stay. `extension=sp.I` allows $i$, which splits $x^2 + 4$. `full=True` uses symbolic roots of any denominator; `.doit()` evaluates them.

In [ ]:
import sympy as sp

x = sp.symbols("x")
R = (x + 3) / ((x - 1) * (x**2 + 4))
print("over Q:        ", sp.apart(R, x))
print("over Q(i):     ", sp.apart(R, x, extension=sp.I))
print("full=True:     ", sp.apart(R, x, full=True).doit())

print()
print("1/(x^2+1)^2:   ", sp.apart(1 / (x**2 + 1) ** 2, x, extension=sp.I))
print("x^3/(x^2+1):   ", sp.apart(x**3 / (x**2 + 1), x, extension=sp.I))
print("1/(x^3+1):     ", sp.apart(1 / (x**3 + 1), x, full=True).doit())

The outputs agree with Worked Examples 2, 3 and 4. For $\frac{1}{x^3 + 1}$ the coefficients are $\frac13$ at $-1$ and $-\frac{1}{3}e^{\pm i\pi/3}$ at $e^{\pm i\pi/3}$, matching the pattern $A_k = \frac{1}{3z_k^2} = -\frac{z_k}{3}$ (since $z_k^3 = -1$).

### 4.3 Repeated poles from the derivative formula

We implement 2.4 with sympy: $A_{k,m-j} = \frac{1}{j!}\frac{d^j}{dz^j}\big[(z - z_k)^m R(z)\big]_{z = z_k}$.

In [ ]:
import sympy as sp

z = sp.symbols("z")


def principal_part(R, pole, order):
    g = sp.cancel((z - pole) ** order * R)
    terms = 0
    for j in range(order):
        coefficient = sp.simplify(sp.diff(g, z, j).subs(z, pole) / sp.factorial(j))
        terms += coefficient / (z - pole) ** (order - j)
    return terms


R = 1 / (z**2 + 1) ** 2
decomposition = principal_part(R, sp.I, 2) + principal_part(R, -sp.I, 2)
print("principal part at i :", principal_part(R, sp.I, 2))
print("principal part at -i:", principal_part(R, -sp.I, 2))
print("sum - R simplifies to:", sp.simplify(decomposition - R))

### 4.4 Recombining conjugate pairs into real terms

A small function implements $B = 2\operatorname{Re}(A)$, $C = -2\operatorname{Re}(A\bar w)$, and we check the real form against the original function on the real line.

In [ ]:
import numpy as np


def recombine(A, w):
    # A/(x - w) + conj(A)/(x - conj(w)) = (B x + C) / (x^2 + p x + q)
    B = 2 * (A.real)
    C = -2 * (A * np.conj(w)).real
    p = -2 * w.real + 0.0          # "+ 0.0" turns -0.0 into 0.0 for display
    q = abs(w) ** 2
    return B, C, p, q


A = -0.4 - 0.05j
w = 2j
B, C, p, q = recombine(A, w)
print(f"({B:+.3f} x {C:+.3f}) / (x^2 {p:+.3f} x {q:+.3f})")

x = np.linspace(-5, 5, 11)
x = x[np.abs(x - 1) > 1e-9]                     # avoid the real pole at x = 1
original = (x + 3) / ((x - 1) * (x**2 + 4))
real_form = 0.8 / (x - 1) + (B * x + C) / (x**2 + p * x + q)
print("real form matches R(x):", np.allclose(original, real_form))

### 4.5 A real integral through complex logarithms

Compute $\displaystyle I = \int_0^1 \frac{dx}{x^3 + 1}$ three ways: complex partial fractions with $\operatorname{Log}$ (2.6), sympy, and the trapezoid rule. The exact value is $\frac{\ln 2}{3} + \frac{\pi}{3\sqrt3}$.

In [ ]:
import numpy as np
import sympy as sp

if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

poles = np.roots([1, 0, 0, 1])                       # -1, e^{+i pi/3}, e^{-i pi/3}
coefficients = 1 / (3 * poles**2)                     # P(z_k)/Q'(z_k)
# np.log is the principal complex logarithm; x - z_k never crosses the negative real axis for x in [0, 1]
via_complex_logs = np.sum(coefficients * (np.log(1 - poles) - np.log(0 - poles)))

x = sp.symbols("x")
via_sympy = sp.integrate(1 / (x**3 + 1), (x, 0, 1))

grid = np.linspace(0, 1, 100001)
via_trapezoid = np.trapezoid(1 / (grid**3 + 1), grid)

exact = np.log(2) / 3 + np.pi / (3 * np.sqrt(3))
print("complex partial fractions:", via_complex_logs)
print("sympy:                    ", sp.nsimplify(via_sympy), "=", float(via_sympy))
print("trapezoid rule:           ", via_trapezoid)
print("ln(2)/3 + pi/(3 sqrt 3):  ", exact)
print("imaginary part is rounding noise:", abs(via_complex_logs.imag) < 1e-12)
print("all agree:", np.isclose(via_complex_logs.real, exact) and np.isclose(via_trapezoid, exact))

## 5. Visualizations

### 5.1 Poles and their coefficients

For $R(z) = \frac{z + 3}{(z - 1)(z^2 + 4)}$, each pole is drawn with an arrow showing its coefficient $A_k$ as a vector (scaled by 2 for visibility). The conjugate poles $\pm 2i$ carry mirror-image arrows: $A_{\bar w} = \overline{A_w}$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

denominator = np.polymul([1, -1], [1, 0, 4])
poles = np.roots(denominator)
coefficients = np.polyval([1, 3], poles) / np.polyval(np.polyder(denominator), poles)
scale = 2.0

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(poles.real, poles.imag, marker="x", color="black", s=120, linewidths=2, zorder=3, label="poles")
for pole, A in zip(poles, coefficients):
    ax.annotate("", xy=(pole.real + scale * A.real, pole.imag + scale * A.imag), xytext=(pole.real, pole.imag),
                arrowprops=dict(arrowstyle="->", color="tab:red", linewidth=2))
    ax.text(pole.real + 0.15, pole.imag + 0.25, f"$A = {A.real:.2f} {A.imag:+.2f}i$", color="tab:red", fontsize=11)
ax.plot([], [], color="tab:red", label="coefficient $A_k$ (x2)")

ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlim(-2.5, 3)
ax.set_ylim(-3, 3)
ax.set_aspect("equal")
ax.set_title("Poles of $(z+3)/((z-1)(z^2+4))$ and their coefficients")
ax.set_xlabel("Re")
ax.set_ylabel("Im")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower left")
plt.show()

### 5.2 Removing a principal part removes the pole

Left: $\log_{10}|R(z)|$, with bright spikes at the three poles. Right: the same after subtracting the single term $\frac{A_{2i}}{z - 2i}$. The spike at $2i$ is gone: near a simple pole, $R$ *is* $\frac{A_k}{z - z_k}$ plus something tame.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

numerator = [1, 3]
denominator = np.polymul([1, -1], [1, 0, 4])
A_2i = (3 + 2j) / ((2j - 1) * 4j)

x = np.linspace(-3, 3, 500)
X, Y = np.meshgrid(x, x)
Z = X + 1j * Y
with np.errstate(divide="ignore", invalid="ignore"):
    R = np.polyval(numerator, Z) / np.polyval(denominator, Z)
    remainder = R - A_2i / (Z - 2j)
panels = [(R, "$\\log_{10}|R(z)|$"), (remainder, "$\\log_{10}|R(z) - A_{2i}/(z - 2i)|$")]

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, (values, title) in zip(axes, panels):
    image = ax.imshow(np.log10(np.abs(values) + 1e-12), extent=(-3, 3, -3, 3), origin="lower",
                      cmap="magma", vmin=-1.5, vmax=1.5)
    ax.scatter([1, 0, 0], [0, 2, -2], marker="x", color="cyan", s=60)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    fig.colorbar(image, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

### 5.3 On the real line: the real pieces add up

The real decomposition of Example 2 on the real axis: the simple-pole term $\frac{4/5}{x - 1}$ carries the blow-up at $x = 1$, while the recombined conjugate pair $\frac{-\frac45x + \frac15}{x^2 + 4}$ is smooth on $\mathbb{R}$ (its poles $\pm 2i$ are off the axis) but shapes the curve.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(-6, 6, 2000)
x = np.where(np.abs(x - 1) < 0.02, np.nan, x)       # break the line at the pole
simple_term = 0.8 / (x - 1)
quadratic_term = (-0.8 * x + 0.2) / (x**2 + 4)
R = (x + 3) / ((x - 1) * (x**2 + 4))

plt.figure(figsize=(9, 5.5))
plt.plot(x, R, color="black", linewidth=2.5, label="$R(x) = (x+3)/((x-1)(x^2+4))$")
plt.plot(x, simple_term, color="tab:blue", linestyle="--", label="$\\frac{4/5}{x - 1}$")
plt.plot(x, quadratic_term, color="tab:orange", linestyle="--", label="$\\frac{-4x/5 + 1/5}{x^2 + 4}$ (pair $\\pm 2i$)")
plt.axhline(0, color="black", linewidth=0.6)
plt.axvline(1, color="gray", linestyle=":", linewidth=1)
plt.ylim(-3, 3)
plt.title("Real partial fractions = complex terms with conjugate pairs recombined")
plt.xlabel("x")
plt.ylabel("y")
plt.grid(True, alpha=0.3)
plt.legend(loc="lower left")
plt.show()

### 5.4 Why the logarithms in 4.5 are safe

For $x \in [0, 1]$, plot the paths $x - z_k$ traced in the plane for the three poles of $\frac{1}{x^3 + 1}$. Each path is a horizontal segment; none touches the branch cut of $\operatorname{Log}$ (the negative real axis, red), so $\operatorname{Log}(x - z_k)$ is a smooth antiderivative along every path.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

poles = np.roots([1, 0, 0, 1])
x = np.linspace(0, 1, 50)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([-2.5, 0], [0, 0], color="tab:red", linewidth=4, alpha=0.6, label="branch cut of Log")
for pole, color in zip(poles, ["tab:blue", "tab:green", "tab:purple"]):
    path = x - pole
    ax.plot(path.real, path.imag, color=color, linewidth=3,
            label=f"$x - z_k$, $z_k = {pole.real:.2f} {pole.imag:+.2f}i$")
    ax.annotate("", xy=(path[-1].real, path[-1].imag), xytext=(path[-8].real, path[-8].imag),
                arrowprops=dict(arrowstyle="->", color=color, linewidth=2))

ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect("equal")
ax.set_title("Paths $x - z_k$ for $x \\in [0, 1]$ avoid the cut")
ax.set_xlabel("Re")
ax.set_ylabel("Im")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower left", fontsize=9)
plt.show()

## 6. Exercises

### Basic

1. Decompose $\dfrac{1}{z^2 + 4}$ over $\mathbb{C}$.
2. Use the cover-up rule to decompose $\dfrac{1}{z(z - i)}$.
3. Decompose $\dfrac{1}{z^2 + 2z + 2}$ over $\mathbb{C}$. (Find the poles first.)

### Intermediate

4. **Draw it first, then compute.** For $R(z) = \dfrac{1}{z^4 - 1}$, sketch the four poles. Using $A_k = \frac{1}{Q'(z_k)} = \frac{z_k}{4}$ (why?), draw each coefficient as an arrow at its pole *before* calculating anything. Then compute the decomposition and compare.
5. Recombine $\dfrac{1 + i}{x - 2i} + \dfrac{1 - i}{x + 2i}$ into a single real fraction.
6. Decompose $\dfrac{1}{z(z - i)^2}$, using the derivative formula for the double pole.

### Advanced

7. Evaluate $\displaystyle\int_0^1 \frac{dx}{x^2 + 1}$ using Example 1 and complex logarithms (principal values), and confirm that it equals $\frac{\pi}{4}$.
8. Prove the formula $A_k = P(z_k)/Q'(z_k)$ for a simple pole, starting from $A_k = \lim_{z \to z_k}(z - z_k)R(z)$.
9. Show that for $\omega = e^{2\pi i/n}$, $\displaystyle\sum_{k=0}^{n-1} \frac{1}{z - \omega^k} = \frac{nz^{n-1}}{z^n - 1}$. (Hint: this is $Q'/Q$ for $Q = z^n - 1$; compute the partial fractions of $Q'/Q$.) Check numerically for $n = 5$.

## 7. Solutions

1. Poles $\pm 2i$; $Q' = 2z$. $A_{2i} = \frac{1}{4i} = -\frac{i}{4}$, $A_{-2i} = \frac{i}{4}$: $\dfrac{1}{z^2 + 4} = \dfrac{-i/4}{z - 2i} + \dfrac{i/4}{z + 2i}$.
2. At $0$: cover $z$, evaluate $\frac{1}{z - i}$ at $0$: $\frac{1}{-i} = i$. At $i$: cover $(z - i)$, evaluate $\frac1z$ at $i$: $\frac1i = -i$. So $\dfrac{1}{z(z - i)} = \dfrac{i}{z} - \dfrac{i}{z - i}$.
3. $z^2 + 2z + 2 = (z + 1)^2 + 1$, poles $-1 \pm i$. $Q'(z) = 2z + 2$, so $A_{-1+i} = \frac{1}{2i} = -\frac{i}{2}$ and $A_{-1-i} = \frac{i}{2}$: $\dfrac{1}{z^2 + 2z + 2} = \dfrac{-i/2}{z + 1 - i} + \dfrac{i/2}{z + 1 + i}$.
4. Poles $1, i, -1, -i$. $Q'(z) = 4z^3$ and $z_k^4 = 1$, so $\frac{1}{4z_k^3} = \frac{z_k}{4z_k^4} = \frac{z_k}{4}$: each arrow points radially **outward** from its pole, with length $\frac14$. Decomposition: $\dfrac{1}{z^4 - 1} = \dfrac{1/4}{z - 1} - \dfrac{1/4}{z + 1} + \dfrac{i/4}{z - i} - \dfrac{i/4}{z + i}$.
5. $A = 1 + i$, $w = 2i$: $B = 2\operatorname{Re}(A) = 2$; $A\bar w = (1 + i)(-2i) = 2 - 2i$, so $C = -2\operatorname{Re}(A\bar w) = -4$. Result: $\dfrac{2x - 4}{x^2 + 4}$. Check: $(1 + i)(x + 2i) + (1 - i)(x - 2i) = 2x + 2i(1 + i) - 2i(1 - i) = 2x + 2i \cdot 2i = 2x - 4$ ✓.
6. At $0$ (simple): $A_0 = \frac{1}{(0 - i)^2} = \frac{1}{-1} = -1$. At $i$ (double): $g(z) = \frac1z$, $A_{i,2} = g(i) = \frac1i = -i$ and $A_{i,1} = g'(i) = -\frac{1}{i^2} = 1$. So $\dfrac{1}{z(z - i)^2} = -\dfrac1z + \dfrac{1}{z - i} - \dfrac{i}{(z - i)^2}$.
7. $\int_0^1 \frac{dx}{x^2 + 1} = \frac{1}{2i}\Big[\operatorname{Log}(x - i) - \operatorname{Log}(x + i)\Big]_0^1$. With $\operatorname{Log}(1 \mp i) = \ln\sqrt2 \mp \frac{i\pi}{4}$ and $\operatorname{Log}(\mp i) = \mp\frac{i\pi}{2}$: the bracket at $1$ is $-\frac{i\pi}{2}$, at $0$ it is $-i\pi$. Difference $\frac{i\pi}{2}$, and $\frac{1}{2i} \cdot \frac{i\pi}{2} = \frac{\pi}{4}$ ✓.
8. Write $Q(z) = (z - z_k)\tilde Q(z)$. Then $(z - z_k)R(z) = \frac{P(z)}{\tilde Q(z)} \to \frac{P(z_k)}{\tilde Q(z_k)}$, since $\tilde Q(z_k) \ne 0$ for a simple pole. By the product rule $Q'(z) = \tilde Q(z) + (z - z_k)\tilde Q'(z)$, so $Q'(z_k) = \tilde Q(z_k)$ and $A_k = \frac{P(z_k)}{Q'(z_k)}$.
9. $Q'/Q$ is proper with simple poles $\omega^k$, and by cover-up its coefficients are $\frac{Q'(\omega^k)}{Q'(\omega^k)} = 1$. Hence $\frac{nz^{n-1}}{z^n - 1} = \sum_k \frac{1}{z - \omega^k}$. (More generally, $\frac{Q'}{Q} = \sum_k \frac{m_k}{z - z_k}$: the logarithmic derivative counts roots with multiplicity, the idea behind the argument principle in module 06.)

Numerical checks:

In [ ]:
import numpy as np
import sympy as sp

z = sp.symbols("z")
print("Ex 1:", sp.apart(1 / (z**2 + 4), z, extension=sp.I))
print("Ex 2:", sp.apart(1 / (z * (z - sp.I)), z))
print("Ex 3:", sp.apart(1 / (z**2 + 2 * z + 2), z, extension=sp.I))
print("Ex 4:", sp.apart(1 / (z**4 - 1), z, extension=sp.I))
print("Ex 5:", sp.simplify((1 + sp.I) / (z - 2 * sp.I) + (1 - sp.I) / (z + 2 * sp.I)))
print("Ex 6:", sp.apart(1 / (z * (z - sp.I) ** 2), z))

integral = (1 / 2j) * ((np.log(1 - 1j) - np.log(1 + 1j)) - (np.log(-1j) - np.log(1j)))
print("Ex 7:", integral, " pi/4 =", np.pi / 4, " close:", np.isclose(integral, np.pi / 4))

n = 5
omega = np.exp(2j * np.pi / n)
test_point = 0.7 - 1.3j
left = sum(1 / (test_point - omega**k) for k in range(n))
right = n * test_point ** (n - 1) / (test_point**n - 1)
print("Ex 9:", np.round(left, 12), np.round(right, 12), " close:", np.isclose(left, right))

## 8. Conclusion

- Over $\mathbb{C}$ every proper rational function is a unique sum of terms $\frac{A_{k,j}}{(z - z_k)^j}$: the FTA removes the need for quadratic terms.
- Simple poles: $A_k = \lim (z - z_k)R(z) = \frac{P(z_k)}{Q'(z_k)}$ (cover-up; the **residue** of module 06). Repeated poles: Taylor coefficients of $(z - z_k)^m R(z)$.
- Real rational functions have conjugate poles with conjugate coefficients; each pair recombines into $\frac{2\operatorname{Re}(A)\,x - 2\operatorname{Re}(A\bar w)}{x^2 - 2\operatorname{Re}(w)\,x + |w|^2}$, the familiar real term.
- `sp.apart(R, x)` works over $\mathbb{Q}$; `extension=sp.I` (or other algebraic numbers) splits further; `full=True` with `.doit()` gives the complete complex decomposition.
- Integrals of rational functions reduce to complex logarithms $\operatorname{Log}(x - z_k)$; the arctangent is the combination of two conjugate logarithms.

**Next:** module 03 begins with [Complex functions as maps](../03_complex_functions/01_complex_functions_as_maps.ipynb): instead of solving $p(z) = 0$, we look at how functions like $z^2$, $1/z$ and $e^z$ transform the whole plane, which is the picture needed to understand the complex logarithm and its branch cut properly.